# Fundamental evaluation metrics

Here we look at simple examples demonstrating classic evaluation metrics for language models.  The examples are kept simple so that we can look explicitly at the steps that go into calculating these.

* Accuracy, precision, recall, and F1 score are classic classification metrics that tell us
  * how often a language model’s predictions are correct overall (accuracy),
  * how reliable its positive predictions are (precision),
  * how many of the true positives it actually finds (recall),
  * how well it balances precision and recall (F1)
* Cross-entropy measures how well the model’s predicted probability distribution matches the true distribution, and is especially important for training and comparing generative models.
* For generation quality, BLEU and ROUGE compare model outputs to reference texts using overlapping n-grams:
  * BLEU is more common in machine translation
  * ROUGE is popular in summarization
* Newer metrics like BERTScore go further by comparing meaning using contextual embeddings rather than just exact word overlap, helping capture semantic similarity even when wording differs.

Together, these metrics provide complementary views of model performance, from raw correctness and probability quality to fluency and semantic faithfulness, which is crucial for building and selecting reliable LLMs.

## Accuracy, precision, recall, F1

First we set up our "data": Ground truth labels and model predictions (1 = positive, 0 = negative)

In [ ]:
y_true = [1, 0, 1, 1, 0, 1]
y_pred = [1, 0, 0, 1, 0, 1]

We can be explicitly formulaic with:
* fp: false positives -> prediction of positive is incorrect
* tp: true positives -> prediction of positive is correct
* fn: false negatives -> prediction of negative is incorrect
* tn: true negatives -> prediction of negative is correct

In [ ]:
tp = tn = fp = fn = 0

for t, p in zip(y_true, y_pred):
    if t == 1 and p == 1:
        tp += 1
    elif t == 0 and p == 0:
        tn += 1
    elif t == 0 and p == 1:
        fp += 1
    elif t == 1 and p == 0:
        fn += 1

Calculating the metrics (for the positive class = 1); we also take care not to divide by zero.

In [ ]:
accuracy  = (tp + tn) / len(y_true)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

In [ ]:
print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")
print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1-score : {f1:.3f}")

In [ ]:
5/6

Admittedly this is easier to do with library functions.  We might calculate the metrics with simple functions for scikit-learn, for example:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = [1, 0, 1, 1, 0, 1]
y_pred = [1, 0, 0, 1, 0, 1]

print(f"Accuracy : {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred):.3f}")
print(f"Recall   : {recall_score(y_true, y_pred):.3f}")
print(f"F1-score : {f1_score(y_true, y_pred):.3f}")

We can also look at the confusion matrix and a classification report with scikit-learn methods.  This can be particularly useful for multicategorical classification, where we might be interested in these metrics across each class.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
confusion_matrix(y_true, y_pred)

In [ ]:
print(classification_report(y_true, y_pred))

Do note that the assignment of "positive" label to 1 is completely arbitrary.  This metric only requires that we establish which label is positive, and the other (or others) will all be in the non-positive class.

In [ ]:
# Ground truth labels and model predictions
y_true = ["pass", "fail", "pass", "pass", "fail", "pass"]
y_pred = ["pass", "fail", "fail", "pass", "fail", "pass"]

tp = tn = fp = fn = 0

for t, p in zip(y_true, y_pred):
    if t == "pass" and p == "pass":
        tp += 1                      # true positive
    elif t == "fail" and p == "fail":
        tn += 1                      # true negative
    elif t == "fail" and p == "pass":
        fp += 1                      # false positive
    elif t == "pass" and p == "fail":
        fn += 1                      # false negative

accuracy  = (tp + tn) / len(y_true)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP={tp}, TN={tn}, FP={fp}, FN={fn}")
print(f"Accuracy : {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall   : {recall:.3f}")
print(f"F1-score : {f1:.3f}")

With scikit-learn, we would need to be explicit about which class label to consider as the positive for precision, recall, and f1 scores.  (The accuracy has no such need).

In [ ]:
y_true = ["pass", "fail", "pass", "pass", "fail", "pass"]
y_pred = ["pass", "fail", "fail", "pass", "fail", "pass"]

print(f"Accuracy : {accuracy_score(y_true, y_pred):.3f}")
print(f"Precision: {precision_score(y_true, y_pred, pos_label="pass"):.3f}")
print(f"Recall   : {recall_score(y_true, y_pred, pos_label="pass"):.3f}")
print(f"F1-score : {f1_score(y_true, y_pred, pos_label="pass"):.3f}")

# Cross-entropy

Classification uses metrics like accuracy, precision, recall , and F1.

This is different than assessing how well a predicted probability distribution matches a true distribution, which is what classification models actually predict on the way to predicting a discrete class label.  It is also what a language model predicts on the way to predicting a definite word. The model is generating a probability distribution over the vocabulary of words, and we train the model by using a continuously valued loss score that assesses how good the probability distribution is.

We look first at binary cross-entropy (two classes) and then at categorical cross-entroy (>2 classes)

In [ ]:
import math

True labels (0 or 1) for a 4-element dataset:

In [ ]:
y_true = [1, 0, 1, 1]

Predicted probabilities of being in the positive class (1):

In [ ]:
y_pred = [0.9, 0.2, 0.8, 0.7]

Binary cross-entropy (the negative log-likelihood of a binary model):

$BCE = -\frac{1}{n} \Sigma[ y*log(p) + (1-y)*log(1-p) ]$

In [ ]:
epsilon = 1e-15  # to avoid log(0)

bce = 0.0
for y, p in zip(y_true, y_pred):
    # clip p to avoid log(0)
    p = min(max(p, epsilon), 1 - epsilon)
    bce += -(y * math.log(p) + (1 - y) * math.log(1 - p))

bce /= len(y_true)

In [ ]:
print(f"Binary Cross Entropy: {bce:.4f}")

Here are a couple examples to see how the BCE score varies as the model makes better/worse predictions.

In [ ]:
y_true = [1, 0, 1, 1]

y_pred = [0.9, 0.2, 0.8, 0.7]
# y_pred = [0.9, 0.1, 0.89, 0.79]
# y_pred = [0.5, 0.5, 0.5, 0.5]
# y_pred = [0.99, 0.01, 0.99, 0.99]
# y_pred = [1, 0, 1, 1]

bce = 0.0
for y, p in zip(y_true, y_pred):
    p = min(max(p, epsilon), 1 - epsilon)
    bce += -(y * math.log(p) + (1 - y) * math.log(1 - p))
bce /= len(y_true)

bce

Categorical cross-entropy is similar, but now the model is categorical rather than binary:

$CCE = -\frac{1}{n} \Sigma_{n} \Sigma_{i} [y_{n,i} log(p_{n,i})]$

where the sum over i is over the class labels and the sum over n is the sum over data points.

In [ ]:
# True class indices for each sample (3 classes: 0, 1, 2)
y_true = [0, 2, 1]   # sample 1 -> class 0, sample 2 -> class 2, sample 3 -> class 1

In [ ]:
# Predicted probabilities for each class (rows sum to 1)
y_pred = [
    [0.7, 0.2, 0.1],  # prediction for sample 1
    [0.1, 0.2, 0.7],  # prediction for sample 2
    [0.2, 0.5, 0.3],  # prediction for sample 3
]

In [ ]:
epsilon = 1e-15  # to avoid log(0)

loss = 0.0
for true_class, probs in zip(y_true, y_pred):
    p = probs[true_class]
    # clip p to avoid log(0)
    p = min(max(p, epsilon), 1 - epsilon)
    loss += -math.log(p)

loss /= len(y_true)

print(f"Categorical Cross Entropy: {loss:.4f}")

In [ ]:
def cce(y_true, y_pred):
    epsilon = 1e-15
    loss = 0.0
    for true_class, probs in zip(y_true, y_pred):
        p = probs[true_class]
        p = min(max(p, epsilon), 1 - epsilon)
        loss += -math.log(p)    
    loss /= len(y_true)
    print(loss)

In [ ]:
y_true = [0, 2, 1]

y_pred = [
    [0.7, 0.2, 0.1],  # prediction for sample 1
    [0.1, 0.2, 0.7],  # prediction for sample 2
    [0.2, 0.5, 0.3],  # prediction for sample 3
]
# y_pred = [
#     [0.9, 0.05, 0.05],  # prediction for sample 1
#     [0.05, 0.05, 0.9],  # prediction for sample 2
#     [0.05, 0.9, 0.05],  # prediction for sample 3
# ]
# y_pred = [
#     [1, 0, 0],  # prediction for sample 1
#     [0, 0, 1],  # prediction for sample 2
#     [0, 1, 0],  # prediction for sample 3
# ]
# y_pred = [
#     [0, 1, 0],  # prediction for sample 1
#     [0, 1, 0],  # prediction for sample 2
#     [1, 0, 0],  # prediction for sample 3
# ]

cce(y_true, y_pred)

# BLEU

BLEU (Bilingual Evaluation Understudy) is a widely used automatic metric for evaluating machine translation and other text generation systems. It works by comparing the system’s output to one or more human reference texts, measuring how many overlapping n-grams (short sequences of words) they share. 

BLEU combines these n-gram precision scores and applies a brevity penalty so very short outputs aren’t unfairly rewarded. Although it’s fast and easy to compute, BLEU doesn’t fully capture meaning or fluency.

Here we will implement:
* BLEU-1 = BP x precision
* BP = brevity penalty to punish very short candidates.
* precision = clipped overlap of unigrams / candidate length

### BLEU‑1 (unigram BLEU)

In [ ]:
import math
from collections import Counter

Our reference and candidate sentences (tokenized here with `split()`):

In [ ]:
reference = "the cat is on the mat".split()
candidate = "the cat sat on the mat".split()

In [ ]:
reference

In [ ]:
candidate

In order to count unigrams in reference and candidate:

In [ ]:
ref_counts = Counter(reference)
cand_counts = Counter(candidate)

In [ ]:
ref_counts

In [ ]:
cand_counts

Here we calculated the clipped unigram precision.

In [ ]:
clipped_count = 0
for word, count in cand_counts.items():
    clipped_count += min(count, ref_counts.get(word, 0))

In [ ]:
clipped_count

And divide by the candidate length to get the precision:

In [ ]:
precision = clipped_count / len(candidate)

In [ ]:
precision

In [ ]:
5/6

Here we define `bp`, the brevity penalty (test this out for a variety of candidates)

In [ ]:
ref_len = len(reference)
cand_len = len(candidate)

if cand_len > ref_len:
    bp = 1.0
else:
    bp = math.exp(1 - ref_len / cand_len)

In [ ]:
bp

In [ ]:
bleu_1 = bp * precision

In [ ]:
print(f"BLEU-1 score: {bleu_1:.4f}")

### using `nltk`’s BLEU

In [ ]:
# pip install nltk  # if needed for your environment

from nltk.translate.bleu_score import sentence_bleu

In [ ]:
reference = "the cat is on the mat".split()
candidate = "the cat sat on the mat".split()

In [ ]:
score = sentence_bleu([reference], candidate)  # default uses 4-grams
# score = sentence_bleu([reference], candidate, (1,))  # use weighting for other n-grams
# score = sentence_bleu([reference], candidate, (1/2,1/2))  # use weighting for other n-grams

print(f"BLEU score: {score:.4f}")

The example above shows the basic idea (unigram BLEU).  If we want to extend to BLEU-2, BLEU-3, etc, then we take the geometric means of unigram precision with bi-gram precision, and with tri-gram precision, and etc.

In [ ]:
reference = "the cat is on the mat".split()
candidate = "the cat sat on the mat".split()

In [ ]:
def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

In [ ]:
Counter(ngrams(reference, 1))

In [ ]:
Counter(ngrams(reference, 2))

In [ ]:
Counter(ngrams(candidate, 2))

In [ ]:
Counter(ngrams(reference, 3))

In [ ]:
def clipped_precision(reference, candidate, n):
    ref_counts  = Counter(ngrams(reference, n))
    cand_counts = Counter(ngrams(candidate, n))

    overlap = 0
    total = sum(cand_counts.values())

    for ng, count in cand_counts.items():
        overlap += min(count, ref_counts.get(ng, 0))

    return overlap / total if total > 0 else 0.0

In [ ]:
# unigram and bigram precisions
p1 = clipped_precision(reference, candidate, 1)
p2 = clipped_precision(reference, candidate, 2)

In [ ]:
p1, p2

In [ ]:
# brevity penalty
ref_len = len(reference)
cand_len = len(candidate)

if cand_len > ref_len:
    bp = 1.0
else:
    bp = math.exp(1 - ref_len / cand_len)

In [ ]:
# BLEU-2: geometric mean of p1 and p2, then multiply by BP
if p1 == 0 or p2 == 0:
    bleu2 = 0.0
else:
    bleu2 = bp * math.exp(0.5 * (math.log(p1) + math.log(p2)))

print(f"Unigram precision (p1): {p1:.4f}")
print(f"Bigram precision  (p2): {p2:.4f}")
print(f"BLEU-2 score      : {bleu2:.4f}")

# ROUGE

BLEU (Bilingual Evaluation Understudy) is precision-oriented.  It measures how many n-grams in the generated text appear in the reference text. It penalizes short outputs with a brevity penalty and is primarily used for machine translation. 

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) is recall-oriented.  It measures how many n-grams from the reference text appear in the generated text. It emphasizes comprehensiveness and is the standard metric for text summarization.

* BLEU is good to use when precision and exact wording matter (e.g., translation). 
* ROUGE is good to use when capturing key content from the reference is critical (e.g., summarization).

Reference (gold) summary and candidate (generated) summary:

In [ ]:
reference = "the cat is on the mat"
candidate = "the cat is on mat"

Tokenize (very naive: just split on spaces)

In [ ]:
ref_tokens = reference.split()
cand_tokens = candidate.split()

Count unigrams

In [ ]:
ref_counts = Counter(ref_tokens)
cand_counts = Counter(cand_tokens)

Overlap of unigrams (clipped counts)

In [ ]:
overlap = 0
for word, count in cand_counts.items():
    overlap += min(count, ref_counts.get(word, 0))

In [ ]:
# ROUGE-1 precision, recall, F1
precision = overlap / len(cand_tokens) if cand_tokens else 0
recall    = overlap / len(ref_tokens)  if ref_tokens else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Overlap unigram count:", overlap)
print(f"ROUGE-1 Precision: {precision:.4f}")
print(f"ROUGE-1 Recall   : {recall:.4f}")
print(f"ROUGE-1 F1       : {f1:.4f}")

This computes ROUGE‑1 (based on unigrams):

* Precision: how much of the candidate is in the reference
* Recall: how much of the reference is covered by the candidate
* F1: harmonic mean of precision and recall

To extend to ROUGE‑2, you’d do the same thing but with bigrams instead of unigrams.

In practice, one might additionally report ROUGE‑L, based on longest common subsequence (captures in‑order matches even if not contiguous), and usually with recall and/or F1.

# BERTScore

BLEU and ROUGE are classic metrics, but they look at exact matching.  An LLM output might be good enough for our purposes if it outputs something with exactly similar meaning without needing to have exactly similar overlapping wordage.

In [ ]:
from transformers import logging
logging.set_verbosity_error()  # Suppresses all logs except errors

In [ ]:
# Install these first in your terminal / notebook:
# pip install torch transformers

import torch
from transformers import AutoTokenizer, AutoModel

# Load a small-ish English BERT model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", verbose=False)
model = AutoModel.from_pretrained("bert-base-uncased")
#tokenizer = AutoTokenizer.from_pretrained("roberta-large")
#model = AutoModel.from_pretrained("roberta-large")

model.eval()

def get_token_embeddings(text):

    encoded = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        out = model(**encoded)

    embeddings = out.last_hidden_state[0]
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])

    # Keep only non-special tokens (drops [CLS], [SEP], [PAD], etc.)
    keep_indices = [
        i for i, tok in enumerate(tokens)
        if tok not in tokenizer.all_special_tokens
    ]
    return embeddings[keep_indices]

In [ ]:
candidate  = "the cat is on the mat"
reference  = "there is a cat on the mat"

In [ ]:
get_token_embeddings(candidate)

In [ ]:
get_token_embeddings(candidate).shape

In [ ]:
cand_emb = get_token_embeddings(candidate)
ref_emb  = get_token_embeddings(reference)

In [ ]:
cand_emb[0] @ cand_emb[0]

In [ ]:
cand_emb = torch.nn.functional.normalize(cand_emb, p=2, dim=1)
ref_emb  = torch.nn.functional.normalize(ref_emb,  p=2, dim=1)

In [ ]:
cand_emb[0] @ cand_emb[0]

In [ ]:
sim = cand_emb @ ref_emb.T

In [ ]:
sim

In [ ]:
sim.max(dim=1)

In [ ]:
sim.max(dim=1).values.mean().item()

In [ ]:
sim.max(dim=0).values.mean().item()

In [ ]:
precision = sim.max(dim=1).values.mean().item()
recall = sim.max(dim=0).values.mean().item()
f1 = 2 * precision * recall / (precision + recall + 1e-8)

precision, recall, f1

In [ ]:
def bertscore(cand, ref):

    cand_emb = get_token_embeddings(cand)
    ref_emb  = get_token_embeddings(ref)

    cand_emb = torch.nn.functional.normalize(cand_emb, p=2, dim=1)
    ref_emb  = torch.nn.functional.normalize(ref_emb,  p=2, dim=1)

    sim = cand_emb @ ref_emb.T

    precision = sim.max(dim=1).values.mean().item()
    recall = sim.max(dim=0).values.mean().item()
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1

This is a **simplified** BERTScore:

* no IDF weighting
* no baseline rescaling
* just raw cosine similarities + greedy matching

But it shows the mechanics explicitly and is easy to tweak.


Here is an easy-to-use library version:

In [ ]:
# !pip install bert-score

In [ ]:
from bert_score import score

Candidate (model output) and reference (gold text)

In [ ]:
cands = ["the cat is on the mat"]
refs  = ["there is a cat on the mat"]

Compute BERTScore (P, R, F1) using an English model

In [ ]:
P, R, F1 = score(cands, refs, lang="en")

print(f"Precision: {P.mean().item():.4f}")
print(f"Recall   : {R.mean().item():.4f}")
print(f"F1       : {F1.mean().item():.4f}")

In [ ]:
from bert_score import BERTScorer

In [ ]:
scorer = BERTScorer(lang='en')

In [ ]:
scorer.model_type

In [ ]:
cands = ["The train was stopped due to a signal failure."]
refs  = ["The signal was red, so the train couldn't proceed."]

P, R, F1 = scorer.score(cands, refs)

print(f"Precision: {P.mean().item():.4f}")
print(f"Recall   : {R.mean().item():.4f}")
print(f"F1       : {F1.mean().item():.4f}")

## In comparision with ROUGE and BLEU:

In [ ]:
ref_tokens = refs[0].split()
cand_tokens = cands[0].split()

ref_counts = Counter(ref_tokens)
cand_counts = Counter(cand_tokens)

overlap = 0
for word, count in cand_counts.items():
    overlap += min(count, ref_counts.get(word, 0))

# ROUGE-1 precision, recall, F1
precision = overlap / len(cand_tokens) if cand_tokens else 0
recall    = overlap / len(ref_tokens)  if ref_tokens else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("Overlap unigram count:", overlap)
print(f"ROUGE-1 Precision: {precision:.4f}")
print(f"ROUGE-1 Recall   : {recall:.4f}")
print(f"ROUGE-1 F1       : {f1:.4f}")

In [ ]:
sentence_bleu(refs, cands[0])

In [ ]:
sentence_bleu(refs, cands[0], (1,))